# Vehicle Re-ID: обучение EfficientNetV2-S на VeRi-776 (Google Colab)

Этот ноутбук переносит пайплайн обучения из локального проекта в Colab,
чтобы использовать бесплатный GPU.

**Перед запуском:**

1. В меню `Runtime -> Change runtime type` выберите **GPU** (T4 обычно
   доступен на бесплатном тарифе).
2. Заранее заархивируйте вашу папку `data/VeRi` в `VeRi.zip` и загрузите
   её на Google Drive (см. Шаг 2 ниже про то, куда именно положить).
3. Проверьте переменные в блоке **"Конфигурация"** (Шаг 5) — пути на Drive,
   гиперпараметры, число сохраняемых чекпоинтов.

**Важно про надёжность обучения в Colab:**

Бесплатные сессии Colab могут завершиться в любой момент (лимит по
времени, неактивность вкладки, отключение GPU). Поэтому в ноутбуке
реализовано **автоматическое возобновление**: при повторном запуске
он сам найдёт последний сохранённый на Drive чекпоинт и продолжит
обучение с той эпохи, на которой остановился, вместо того чтобы
начинать заново.


## Шаг 1. Проверка GPU

In [1]:
!nvidia-smi

Tue Sep 15 12:49:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Шаг 2. Подключение Google Drive и подготовка данных

**Почему датасет нужно распаковывать в локальное хранилище Colab
(`/content/...`), а не читать напрямую с примонтированного Drive:**

Google Drive монтируется через FUSE (сетевую файловую систему), и
каждое открытие файла — это отдельный сетевой запрос. VeRi-776 состоит
из ~50 000 отдельных jpg-файлов, и чтение каждого из них напрямую с
Drive на каждой итерации DataLoader'а будет **на порядок медленнее**,
чем чтение с локального SSD-хранилища сессии Colab. Поэтому:

  - Сам архив `VeRi.zip` лежит на Drive (это просто один файл, монтирование
    для него не проблема).
  - Мы копируем/распаковываем его один раз в начале сессии в `/content/data`
    (локальный диск виртуальной машины Colab).
  - Все чтения изображений во время обучения идут с локального диска.
  - На Drive остаются только чекпоинты (немного файлов, читаются редко).

Заранее заархивируйте так, чтобы после распаковки получилась структура:

```
VeRi/
    image_train/
    image_query/
    image_test/
```

и загрузите `VeRi.zip` в `MyDrive/VehicleReID/data/VeRi.zip` (путь можно
поменять в конфигурации ниже).


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Поменяйте под свою структуру на Google Drive, если она отличается.
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/VehicleReID'
DATA_ZIP_PATH = f'{DRIVE_PROJECT_ROOT}/data/VeRi.zip'

LOCAL_DATA_ROOT = '/content/data/VeRi'

if not os.path.isdir(LOCAL_DATA_ROOT):
    os.makedirs('/content/data', exist_ok=True)
    print(f'Распаковываю {DATA_ZIP_PATH} -> /content/data ...')
    !unzip -q "{DATA_ZIP_PATH}" -d /content/data
    print('Готово. Содержимое /content/data:', os.listdir('/content/data'))
else:
    print(f'{LOCAL_DATA_ROOT} уже существует, распаковка не требуется.')

# Проверяем, что после распаковки структура именно та, что нужна.
# Если архив распаковался с ЛИШНИМ уровнем вложенности
# (/content/data/VeRi/VeRi/image_train вместо /content/data/VeRi/image_train),
# поправьте LOCAL_DATA_ROOT на дополнительный подкаталог.
for sub in ('image_train', 'image_query', 'image_test'):
    path = os.path.join(LOCAL_DATA_ROOT, sub)
    status = 'OK' if os.path.isdir(path) else 'НЕ НАЙДЕНО'
    print(f'  {path}: {status}')

Распаковываю /content/drive/MyDrive/VehicleReID/data/VeRi.zip -> /content/data ...
Готово. Содержимое /content/data: ['VeRi']
  /content/data/VeRi/image_train: OK
  /content/data/VeRi/image_query: OK
  /content/data/VeRi/image_test: OK


## Шаг 3. Установка зависимостей

В Colab `torch` и `torchvision` обычно уже предустановлены и подобраны
под доступную версию CUDA, поэтому переустанавливать их не нужно —
это может даже сломать совместимость. Просто проверяем версии.

In [4]:
import torch
import torchvision

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA доступна:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
CUDA доступна: True
GPU: Tesla T4


## Шаг 4. Код проекта

Ниже воссоздаётся та же структура модулей, что и в локальном проекте
(`ml/datasets`, `ml/samplers`, `ml/models`, `ml/losses`) — код идентичен
десктопной версии, чтобы не было расхождений между Colab и локальной
разработкой. Каждая ячейка записывает файл через `%%writefile`.

In [5]:
!mkdir -p /content/ml/datasets /content/ml/samplers /content/ml/models /content/ml/losses

In [6]:
%%writefile /content/ml/datasets/veri_dataset.py
"""
Загрузчик датасета VeRi-776 для задачи Vehicle Re-Identification.

Ожидаемая структура каталогов после скачивания VeRi-776:

    VeRi/
        image_train/
        image_query/
        image_test/

Формат имени файла (стандартный релиз VeRi-776):

    {vehicle_id:04d}_c{camera_id:03d}_{frame_id:08d}_{track_idx}.jpg

    пример:  0002_c002_00030600_0.jpg
             -> vehicle_id = 2, camera_id = 2

Модуль строит три сплита (train / query / gallery) в виде списков
VeRiSample(path, pid, camid) и предоставляет обёртку
torch.utils.data.Dataset для каждого из них.

Важно:
    - Для train идентичности перенумеровываются в непрерывный диапазон
      [0, num_train_pids) — это требование classification head
      (CrossEntropyLoss ожидает индексы классов 0..N-1).
    - Для query/gallery исходные vehicle_id сохраняются без изменений,
      т.к. по ним происходит сопоставление query <-> gallery на этапе
      оценки (mAP / CMC).
"""

from __future__ import annotations

import glob
import os
import re
from dataclasses import dataclass
from typing import Callable, List, Optional, Tuple

from PIL import Image
from torch.utils.data import Dataset

# VeRi-776: 0001_c001_00016450_0.jpg
_FILENAME_PATTERN = re.compile(r"^(\d+)_c(\d+)_")


@dataclass
class VeRiSample:
    path: str
    pid: int      # идентичность транспортного средства
    camid: int    # id камеры


def _parse_filename(filename: str) -> Tuple[int, int]:
    """Извлекает (vehicle_id, camera_id) из имени файла VeRi-776."""
    match = _FILENAME_PATTERN.match(os.path.basename(filename))
    if match is None:
        raise ValueError(
            f"Имя файла не соответствует формату VeRi-776: {filename}"
        )
    vehicle_id, camera_id = match.groups()
    return int(vehicle_id), int(camera_id)


def _scan_folder(folder: str, relabel: bool = False) -> List[VeRiSample]:
    """
    Сканирует папку с изображениями и строит список VeRiSample.

    Изображения с pid в (-1, 0) пропускаются (junk/distractor images) —
    в стандартном VeRi-776 их нет, но проверка оставлена для
    совместимости с другими релизами/датасетами похожего формата.
    """
    img_paths = sorted(
        glob.glob(os.path.join(folder, "*.jpg"))
        + glob.glob(os.path.join(folder, "*.png"))
    )
    if not img_paths:
        raise FileNotFoundError(f"Изображения не найдены в {folder}")

    raw_samples = []
    pid_set = set()
    for path in img_paths:
        pid, camid = _parse_filename(path)
        if pid in (-1, 0):
            continue
        raw_samples.append((path, pid, camid))
        pid_set.add(pid)

    if relabel:
        pid2label = {pid: idx for idx, pid in enumerate(sorted(pid_set))}
        samples = [
            VeRiSample(p, pid2label[pid], camid) for p, pid, camid in raw_samples
        ]
    else:
        samples = [VeRiSample(p, pid, camid) for p, pid, camid in raw_samples]

    return samples


class VeRi776:
    """
    Строит train / query / gallery сплиты из корневой директории VeRi-776.
    """

    def __init__(self, root: str):
        self.root = root
        self.train_dir = os.path.join(root, "image_train")
        self.query_dir = os.path.join(root, "image_query")
        self.gallery_dir = os.path.join(root, "image_test")

        for d in (self.train_dir, self.query_dir, self.gallery_dir):
            if not os.path.isdir(d):
                raise FileNotFoundError(
                    f"Ожидаемая подпапка VeRi-776 не найдена: {d}\n"
                    f"  Абсолютный путь, который реально проверялся: {os.path.abspath(d)}\n"
                    f"  Текущая рабочая директория: {os.getcwd()}\n"
                    f"  Проверьте, что --data-root указан относительно "
                    f"той папки, из которой вы запускаете скрипт, "
                    f"либо используйте абсолютный путь."
                )

        self.train = _scan_folder(self.train_dir, relabel=True)
        self.query = _scan_folder(self.query_dir, relabel=False)
        self.gallery = _scan_folder(self.gallery_dir, relabel=False)

        self.num_train_pids = len({s.pid for s in self.train})
        self.num_train_cams = len({s.camid for s in self.train})

    def summary(self) -> str:
        return (
            f"VeRi-776 датасет @ {self.root}\n"
            f"  train:   {len(self.train):>6} изобр. | "
            f"{self.num_train_pids:>4} идентичностей | {self.num_train_cams} камер\n"
            f"  query:   {len(self.query):>6} изобр. | "
            f"{len({s.pid for s in self.query}):>4} идентичностей\n"
            f"  gallery: {len(self.gallery):>6} изобр. | "
            f"{len({s.pid for s in self.gallery}):>4} идентичностей"
        )


class VeRiImageDataset(Dataset):
    """
    torch Dataset, оборачивающий список VeRiSample.

    Возвращает (image_tensor, pid, camid, path). Путь `path` сохраняется,
    чтобы можно было отследить, какому исходному файлу соответствует
    результат — пригодится для отладки и для отображения найденных
    кандидатов в веб-интерфейсе (top-K similar).
    """

    def __init__(
        self,
        samples: List[VeRiSample],
        transform: Optional[Callable] = None,
    ):
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        sample = self.samples[index]
        img = Image.open(sample.path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, sample.pid, sample.camid, sample.path


if __name__ == "__main__":
    # Быстрая ручная проверка:
    #   python veri_dataset.py /path/to/VeRi
    import sys

    if len(sys.argv) != 2:
        print("Использование: python veri_dataset.py <путь_к_корню_VeRi>")
        sys.exit(1)

    ds = VeRi776(sys.argv[1])
    print(ds.summary())


Writing /content/ml/datasets/veri_dataset.py


In [7]:
%%writefile /content/ml/samplers/pk_sampler.py
"""
PK-sampler для метрического обучения (Batch-Hard Triplet Loss).

Каждый мини-батч формируется из P идентичностей по K изображений
каждая (размер батча = P * K). Это гарантирует, что в каждом батче
для любого якоря найдётся достаточно позитивных примеров для
online hard-mining, что требуется для batch-hard triplet loss
(см. ml/losses/triplet.py).
"""

from __future__ import annotations

import copy
import random
from collections import defaultdict
from typing import Sequence

from torch.utils.data import Sampler


class PKSampler(Sampler):
    """
    Формирует индексы так, что каждый батч содержит `num_instances`
    изображений для каждой из `num_pids_per_batch` случайно выбранных
    идентичностей.

    Args:
        pids: последовательность идентичностей, поэлементно
            соответствующая датасету, с которым будет использоваться
            сэмплер (например, [s.pid for s in dataset.samples]).
        num_pids_per_batch: P — количество различных идентичностей
            на батч.
        num_instances: K — количество изображений на идентичность.
            Идентичности, у которых меньше K изображений, дополняются
            выборкой с повторением до K.

    Итоговый batch_size = num_pids_per_batch * num_instances.
    Сэмплер передаётся в DataLoader через `sampler=`, при этом
    `shuffle=True` в DataLoader указывать не нужно — перемешивание
    делается самим сэмплером.

    Примечание: `len(sampler)` — это оценка ёмкости сверху (сумма по
    идентичностям), а не точное число индексов за один вызов
    `__iter__`. Реальное количество обычно чуть меньше, так как в
    цикле выбора батчей идентичность выбывает из пула, как только
    у неё заканчиваются накопленные "пачки" по K изображений — это
    может произойти раньше, чем у остальных. Не используйте
    len(sampler) для точного расчёта steps_per_epoch, только как
    приблизительный ориентир.
    """

    def __init__(
        self,
        pids: Sequence[int],
        num_pids_per_batch: int,
        num_instances: int,
    ):
        self.pids = list(pids)
        self.num_pids_per_batch = num_pids_per_batch
        self.num_instances = num_instances
        self.batch_size = num_pids_per_batch * num_instances

        self.index_dict = defaultdict(list)
        for index, pid in enumerate(self.pids):
            self.index_dict[pid].append(index)
        self.pid_list = list(self.index_dict.keys())

        if len(self.pid_list) < self.num_pids_per_batch:
            raise ValueError(
                f"num_pids_per_batch ({self.num_pids_per_batch}) больше, чем "
                f"количество доступных идентичностей ({len(self.pid_list)})."
            )

        # Оценка длины эпохи: для каждой идентичности считаем, сколько
        # "полных" наборов размера num_instances из неё можно набрать.
        self.length = 0
        for pid in self.pid_list:
            num_imgs = len(self.index_dict[pid])
            num_imgs = max(num_imgs, self.num_instances)
            self.length += num_imgs - num_imgs % self.num_instances

    def __iter__(self):
        batch_idxs_dict = defaultdict(list)

        for pid in self.pid_list:
            idxs = copy.deepcopy(self.index_dict[pid])
            if len(idxs) < self.num_instances:
                idxs = random.choices(idxs, k=self.num_instances)
            random.shuffle(idxs)

            batch_idxs = []
            for idx in idxs:
                batch_idxs.append(idx)
                if len(batch_idxs) == self.num_instances:
                    batch_idxs_dict[pid].append(batch_idxs)
                    batch_idxs = []

        avail_pids = copy.deepcopy(self.pid_list)
        final_idxs = []

        while len(avail_pids) >= self.num_pids_per_batch:
            selected_pids = random.sample(avail_pids, self.num_pids_per_batch)
            for pid in selected_pids:
                batch_idxs = batch_idxs_dict[pid].pop(0)
                final_idxs.extend(batch_idxs)
                if len(batch_idxs_dict[pid]) == 0:
                    avail_pids.remove(pid)

        return iter(final_idxs)

    def __len__(self):
        return self.length


if __name__ == "__main__":
    # Мини-проверка на синтетических данных:
    # 5 идентичностей, у каждой разное число изображений.
    fake_pids = (
        [0] * 3 + [1] * 7 + [2] * 4 + [3] * 2 + [4] * 5
    )
    sampler = PKSampler(fake_pids, num_pids_per_batch=2, num_instances=4)
    batches = list(sampler)
    print(f"Всего индексов: {len(batches)}, batch_size = {sampler.batch_size}")
    print(f"len(sampler) = {len(sampler)}")


Writing /content/ml/samplers/pk_sampler.py


In [8]:
%%writefile /content/ml/models/efficientnetv2_reid.py
"""
Модель для Vehicle Re-Identification: EfficientNetV2-S backbone +
embedding head + classification head.

Архитектура соответствует пайплайну проекта:

    Crop -> EfficientNetV2-S -> GAP -> Embedding head (512D)
                                          |
                                          +--> Classification head (CE)
                                          |
                                          +--> L2-norm --> Triplet loss

Используется приём "BNNeck" (широко применяется в person/vehicle ReID,
см., например, Luo et al. "Bag of Tricks and A Strong Baseline for
Deep Person Re-identification", 2019):

    - Triplet loss считается на эмбеддинге ДО BatchNorm1d.
    - Classification (ID) loss считается на эмбеддинге ПОСЛЕ BatchNorm1d.

Причина: BatchNorm выравнивает масштаб признаков, что помогает
softmax-классификации, но одновременно "сплющивает" метрическое
пространство, что вредит triplet loss. Разделение этих двух путей
после общего BatchNorm обычно даёт заметный прирост качества по
сравнению с использованием одного и того же вектора для обеих задач.
"""

from __future__ import annotations

from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights


class VehicleReIDNet(nn.Module):
    """
    Args:
        num_classes: количество уникальных идентичностей в train-сплите
            (dataset.num_train_pids из VeRi776). Нужно для classification
            head.
        embedding_dim: размерность итогового эмбеддинга (по умолчанию 512,
            согласно архитектуре проекта).
        pretrained: использовать ли ImageNet-претрейн для backbone.
        dropout_p: вероятность dropout в embedding head.
    """

    def __init__(
        self,
        num_classes: int,
        embedding_dim: int = 512,
        pretrained: bool = True,
        dropout_p: float = 0.5,
    ):
        super().__init__()

        weights = EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
        backbone = efficientnet_v2_s(weights=weights)

        # У efficientnet_v2_s из torchvision структура:
        #   backbone.features   -> свёрточные блоки (то, что нам нужно)
        #   backbone.avgpool    -> глобальный average pooling (GAP)
        #   backbone.classifier -> исходная ImageNet-голова (нам не нужна)
        #
        # Забираем только feature extractor + GAP, отбрасываем classifier.
        self.backbone_features = backbone.features
        self.gap = backbone.avgpool  # nn.AdaptiveAvgPool2d(1)

        # Число каналов на выходе последнего свёрточного блока
        # efficientnet_v2_s (перед классификатором) — 1280.
        backbone_out_dim = backbone.classifier[1].in_features

        # --- Embedding head ---
        # Linear -> BatchNorm1d -> ReLU -> Dropout -> Linear(embedding_dim)
        self.embedding_head = nn.Sequential(
            nn.Linear(backbone_out_dim, backbone_out_dim // 2),
            nn.BatchNorm1d(backbone_out_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(backbone_out_dim // 2, embedding_dim),
        )

        # BNNeck: отдельный BatchNorm1d после эмбеддинга, вход для
        # classification head. Bias отключён — стандартная практика
        # для BNNeck (см. Luo et al., 2019), т.к. смещение съедается
        # последующим Linear-слоем классификатора.
        self.bottleneck = nn.BatchNorm1d(embedding_dim)
        self.bottleneck.bias.requires_grad_(False)

        # --- Classification head ---
        self.classifier = nn.Linear(embedding_dim, num_classes, bias=False)

        self._init_weights()

    def _init_weights(self) -> None:
        """Kaiming-инициализация для новых (не претрейн) слоёв."""
        for m in self.embedding_head:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

        nn.init.constant_(self.bottleneck.weight, 1.0)
        nn.init.normal_(self.classifier.weight, std=0.001)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Backbone + GAP -> плоский вектор признаков (до embedding head)."""
        x = self.backbone_features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return x

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], torch.Tensor]:
        """
        Returns:
            embedding: "сырой" эмбеддинг (до BatchNorm), используется
                для triplet loss после L2-нормализации. Также именно
                этот вектор (нормализованный) следует сохранять в
                pgvector/FAISS для инференса.
            logits: выход classification head (для CrossEntropyLoss),
                считается только в режиме обучения (self.training),
                иначе None — на инференсе классификатор не нужен.
            normalized_embedding: L2-нормализованный embedding —
                именно его нужно использовать для cosine similarity
                search и для triplet loss.
        """
        feat = self.extract_features(x)
        embedding = self.embedding_head(feat)

        normalized_embedding = F.normalize(embedding, p=2, dim=1)

        logits = None
        if self.training:
            bn_embedding = self.bottleneck(embedding)
            logits = self.classifier(bn_embedding)

        return embedding, logits, normalized_embedding


if __name__ == "__main__":
    # Быстрая проверка форм тензоров на случайном батче.
    model = VehicleReIDNet(num_classes=576, pretrained=False)
    model.train()
    dummy = torch.randn(8, 3, 224, 224)
    embedding, logits, norm_embedding = model(dummy)
    print("embedding:", embedding.shape)          # [8, 512]
    print("logits:", logits.shape)                # [8, 576]
    print("normalized_embedding:", norm_embedding.shape)  # [8, 512]

    model.eval()
    with torch.no_grad():
        _, logits_eval, norm_embedding_eval = model(dummy)
    print("logits in eval mode:", logits_eval)     # None


Writing /content/ml/models/efficientnetv2_reid.py


In [9]:
%%writefile /content/ml/losses/triplet.py
"""
Batch-Hard Triplet Loss (Hermans et al., "In Defense of the Triplet
Loss for Person Re-Identification", 2017).

Идея: вместо того чтобы заранее формировать конкретные тройки
(anchor, positive, negative), для каждого anchor'а в батче на лету
выбирается САМЫЙ ТРУДНЫЙ позитив (наиболее удалённый образец той же
идентичности) и САМЫЙ ТРУДНЫЙ негатив (наиболее близкий образец другой
идентичности). Это возможно только если батч построен по схеме P x K
(P идентичностей, K изображений каждая) — именно для этого нужен
PKSampler из ml/samplers/pk_sampler.py.

Loss для одного anchor'а i:
    L_i = max(0, d(a_i, hardest_positive_i) - d(a_i, hardest_negative_i) + margin)

Итоговый loss — среднее по всем anchor'ам в батче.
"""

from __future__ import annotations

from typing import Optional

import torch
import torch.nn as nn


def _pairwise_euclidean_dist(embeddings: torch.Tensor) -> torch.Tensor:
    """
    Матрица попарных евклидовых расстояний между эмбеддингами батча.

    Args:
        embeddings: тензор [N, D].

    Returns:
        dist: тензор [N, N], dist[i, j] = ||e_i - e_j||_2.

    Реализовано через тождество:
        ||a - b||^2 = ||a||^2 + ||b||^2 - 2 * a.b
    вместо прямого попарного вычитания — это заметно быстрее на GPU
    и не требует O(N^2 * D) промежуточной памяти.
    """
    # sq_norms[i] = ||e_i||^2
    sq_norms = (embeddings ** 2).sum(dim=1, keepdim=True)  # [N, 1]

    # dist_sq[i, j] = ||e_i||^2 + ||e_j||^2 - 2 * e_i . e_j
    dist_sq = sq_norms + sq_norms.t() - 2.0 * torch.mm(embeddings, embeddings.t())

    # Из-за ошибок округления могут появиться небольшие отрицательные
    # значения на диагонали (расстояние точки до самой себя) —
    # обрезаем снизу нулём перед взятием корня.
    dist_sq = dist_sq.clamp(min=0.0)

    # Для численной стабильности градиента sqrt в нуле (d/dx sqrt(x) -> inf
    # при x->0) добавляем небольшой эпсилон только там, где значение
    # действительно нулевое.
    eps_mask = (dist_sq == 0.0).float()
    dist = torch.sqrt(dist_sq + eps_mask * 1e-12)

    return dist


class BatchHardTripletLoss(nn.Module):
    """
    Args:
        margin: отступ alpha в формуле triplet loss. Если None,
            используется "soft margin" вариант: log(1 + exp(d_ap - d_an))
            вместо max(0, d_ap - d_an + margin) — это устраняет
            необходимость подбирать margin вручную (см. Hermans et al.,
            раздел про soft-margin), но может сходиться медленнее на
            старте обучения.
    """

    def __init__(self, margin: Optional[float] = 0.3):
        super().__init__()
        self.margin = margin

    def forward(
        self, embeddings: torch.Tensor, labels: torch.Tensor
    ) -> torch.Tensor:
        """
        Args:
            embeddings: [N, D] — эмбеддинги батча (рекомендуется подавать
                L2-нормализованные эмбеддинги, т.е. normalized_embedding
                из VehicleReIDNet.forward).
            labels: [N] — идентичности (pid) для каждого эмбеддинга.
                Батч должен быть сформирован PKSampler'ом, чтобы у
                каждой идентичности было >= 2 образцов (иначе для неё
                не найдётся positive).

        Returns:
            Скалярный тензор — усреднённый по батчу loss.
        """
        dist_mat = _pairwise_euclidean_dist(embeddings)  # [N, N]
        n = embeddings.size(0)

        # is_pos[i, j] = True, если j - позитив для i (тот же pid, i != j)
        # is_neg[i, j] = True, если j - негатив для i (другой pid)
        labels = labels.view(-1, 1)
        same_id = labels.eq(labels.t())  # [N, N], True если labels совпадают

        # Убираем диагональ (i == j) из позитивов - сам с собой не сравниваем.
        identity_mask = torch.eye(n, dtype=torch.bool, device=embeddings.device)
        is_pos = same_id & (~identity_mask)
        is_neg = ~same_id

        # Для hardest positive нам нужен МАКСИМУМ расстояния среди
        # позитивов -> заменяем расстояния до непозитивов на -inf,
        # чтобы они не попали в max.
        dist_ap = dist_mat.masked_fill(~is_pos, float("-inf")).max(dim=1).values

        # Для hardest negative нужен МИНИМУМ расстояния среди
        # негативов -> заменяем расстояния до непозитивов на +inf.
        dist_an = dist_mat.masked_fill(~is_neg, float("inf")).min(dim=1).values

        if torch.isinf(dist_ap).any():
            raise ValueError(
                "У некоторых идентичностей в батче нет позитивной пары "
                "(только 1 изображение на pid). Проверьте, что PKSampler "
                "настроен с num_instances >= 2."
            )

        if self.margin is not None:
            # Классическая формула (уравнение 11 в обзорной статье):
            # max(0, d_ap - d_an + margin)
            loss = torch.clamp(dist_ap - dist_an + self.margin, min=0.0)
        else:
            # Soft-margin вариант: log(1 + exp(d_ap - d_an))
            loss = torch.log1p(torch.exp(dist_ap - dist_an))

        return loss.mean()


if __name__ == "__main__":
    # Быстрая проверка на синтетическом батче P=4 идентичности x K=4.
    torch.manual_seed(0)
    P, K, D = 4, 4, 512
    embeddings = torch.randn(P * K, D)
    embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)  # L2-норм.
    labels = torch.arange(P).repeat_interleave(K)

    criterion = BatchHardTripletLoss(margin=0.3)
    loss = criterion(embeddings, labels)
    print("Batch-hard triplet loss на случайном батче:", loss.item())


Writing /content/ml/losses/triplet.py


In [11]:
# Файлы-маркеры пакетов, чтобы импорт "from ml.datasets.veri_dataset import ..."
# работал так же, как в локальном проекте.
!touch /content/ml/__init__.py
!touch /content/ml/datasets/__init__.py
!touch /content/ml/samplers/__init__.py
!touch /content/ml/models/__init__.py
!touch /content/ml/losses/__init__.py

import sys

# ВАЖНО: добавляем в sys.path родительскую папку /content, а НЕ /content/ml.
#
# Если добавить саму /content/ml, то её подпапки (datasets/, models/,
# samplers/, losses/) станут ВЕРХНЕУРОВНЕВЫМИ именами пакетов Python.
# Имена datasets и models слишком общие: в Colab уже предустановлена
# библиотека HuggingFace `datasets`, и Python находит именно её вместо
# нашей локальной папки, что приводит к
# "ModuleNotFoundError: No module named 'datasets.veri_dataset'"
# (сам пакет datasets находится, а нужного файла внутри него, конечно,
# нет - это чужая библиотека).
#
# Импортируя всё через префикс `ml.` (ml.datasets, ml.models, ...),
# мы гарантируем, что наши модули не пересекаются по имени ни с одной
# стандартной или предустановленной в Colab библиотекой.
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from ml.datasets.veri_dataset import VeRi776, VeRiImageDataset
from ml.samplers.pk_sampler import PKSampler
from ml.models.efficientnetv2_reid import VehicleReIDNet
from ml.losses.triplet import BatchHardTripletLoss

print('Модули проекта успешно импортированы.')

Модули проекта успешно импортированы.


## Шаг 5. Конфигурация

Все гиперпараметры и пути собраны в одном месте — здесь всё, что раньше
передавалось через `argparse` в десктопной версии `train.py`.

**`MAX_CHECKPOINTS_TO_KEEP`** — сколько последних по номеру эпохи
чекпоинтов хранить на Drive одновременно. При сохранении нового
чекпоинта самые старые автоматически удаляются, если их число
превышает этот лимит. Отдельно всегда хранится `best_checkpoint.pt`
(чекпоинт с наименьшим средним loss за эпоху) — он в ротацию не
попадает и не удаляется.

In [10]:
from types import SimpleNamespace

cfg = SimpleNamespace(
    # Данные
    data_root=LOCAL_DATA_ROOT,
    image_size=(224, 224),

    # Батчинг P x K (см. pk_sampler.py)
    num_pids_per_batch=8,
    num_instances=4,
    num_workers=2,          # в Colab обычно 2 CPU-ядра доступно

    # Обучение
    epochs=60,
    lr=3.5e-4,
    weight_decay=5e-4,
    milestones=[30, 50],
    lr_gamma=0.1,
    label_smoothing=0.1,
    triplet_margin=0.3,
    triplet_loss_weight=1.0,
    embedding_dim=512,
    dropout=0.5,
    pretrained=True,

    # Чекпоинты на Google Drive
    checkpoint_dir=f'{DRIVE_PROJECT_ROOT}/checkpoints',
    max_checkpoints_to_keep=3,   # хранить не более 3 "обычных" чекпоинтов
    save_every=1,                # сохранять каждую эпоху (нужно для resume)

    # Прочее
    log_every=50,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    seed=42,

    # Кэш предобученных весов backbone — кладём на Drive, чтобы не
    # перекачивать ~85 МБ заново при каждом перезапуске сессии Colab.
    torch_hub_cache_dir=f'{DRIVE_PROJECT_ROOT}/torch_hub_cache',
)

print(cfg)

namespace(data_root='/content/data/VeRi', image_size=(224, 224), num_pids_per_batch=8, num_instances=4, num_workers=2, epochs=60, lr=0.00035, weight_decay=0.0005, milestones=[30, 50], lr_gamma=0.1, label_smoothing=0.1, triplet_margin=0.3, triplet_loss_weight=1.0, embedding_dim=512, dropout=0.5, pretrained=True, checkpoint_dir='/content/drive/MyDrive/VehicleReID/checkpoints', max_checkpoints_to_keep=3, save_every=1, log_every=50, device='cuda', seed=42, torch_hub_cache_dir='/content/drive/MyDrive/VehicleReID/torch_hub_cache')


## Шаг 6. Кэш весов backbone на Google Drive

То же самое, что вы уже настраивали локально через `--cache-dir`, только
теперь кэш живёт на Drive, а не на диске виртуальной машины Colab
(которая обнуляется при каждом перезапуске среды выполнения).

In [12]:
os.makedirs(cfg.torch_hub_cache_dir, exist_ok=True)
torch.hub.set_dir(cfg.torch_hub_cache_dir)
print('Кэш torch.hub:', torch.hub.get_dir())

Кэш torch.hub: /content/drive/MyDrive/VehicleReID/torch_hub_cache


## Шаг 7. Аугментации (без изменений относительно локальной версии)

In [13]:
from torchvision import transforms

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD = [0.229, 0.224, 0.225]


def build_train_transform(image_size):
    h, w = image_size
    return transforms.Compose([
        transforms.Resize((h, w)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Pad(10),
        transforms.RandomCrop((h, w)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
        transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
    ])

## Шаг 8. Датасет, PK-сэмплер, DataLoader

Логика идентична локальному `train.py`: PKSampler уже отдаёт индексы,
сгруппированные по P×K, поэтому DataLoader используется с `sampler=`
и `batch_size=`, а не с кастомным `batch_sampler=`.

In [14]:
print('Загружаю VeRi-776...')
dataset = VeRi776(cfg.data_root)
print(dataset.summary())

train_transform = build_train_transform(cfg.image_size)
train_dataset = VeRiImageDataset(dataset.train, transform=train_transform)

train_pids = [sample.pid for sample in dataset.train]
pk_sampler = PKSampler(
    pids=train_pids,
    num_pids_per_batch=cfg.num_pids_per_batch,
    num_instances=cfg.num_instances,
)
batch_size = pk_sampler.batch_size

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    sampler=pk_sampler,
    batch_size=batch_size,
    num_workers=cfg.num_workers,
    drop_last=True,
    pin_memory=(cfg.device == 'cuda'),
)

print(
    f'DataLoader готов: batch_size={batch_size} '
    f'(P={cfg.num_pids_per_batch} x K={cfg.num_instances}), '
    f'~{len(pk_sampler) // batch_size} батчей на эпоху (оценка)'
)

Загружаю VeRi-776...
VeRi-776 датасет @ /content/data/VeRi
  train:    37778 изобр. |  576 идентичностей | 20 камер
  query:     1678 изобр. |  200 идентичностей
  gallery:  11579 изобр. |  200 идентичностей
DataLoader готов: batch_size=32 (P=8 x K=4), ~1155 батчей на эпоху (оценка)


## Шаг 9. Модель, функции потерь, оптимизатор, scheduler

In [15]:
torch.manual_seed(cfg.seed)

model = VehicleReIDNet(
    num_classes=dataset.num_train_pids,
    embedding_dim=cfg.embedding_dim,
    pretrained=cfg.pretrained,
    dropout_p=cfg.dropout,
).to(cfg.device)

ce_loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
triplet_loss_fn = BatchHardTripletLoss(margin=cfg.triplet_margin)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=cfg.milestones, gamma=cfg.lr_gamma
)

print('Модель и оптимизатор созданы.')

Модель и оптимизатор созданы.


## Шаг 10. Сохранение чекпоинтов с ротацией и возобновление обучения

Здесь два ключевых механизма:

1. **`save_checkpoint_with_rotation`** — сохраняет чекпоинт текущей эпохи
   на Drive, затем удаляет самые старые "обычные" чекпоинты, если их
   стало больше `cfg.max_checkpoints_to_keep`. Отдельно, если loss текущей
   эпохи — лучший за всё обучение, обновляется `best_checkpoint.pt`
   (никогда не удаляется ротацией).

2. **`try_resume_from_checkpoint`** — при старте ноутбука ищет на Drive
   последний по номеру эпохи чекпоинт и, если находит, загружает в него
   веса модели, состояние оптимизатора и scheduler'а, чтобы продолжить
   обучение, а не начинать с нуля. Это то, что спасает от последствий
   разрыва сессии Colab.

In [16]:
import glob
import re


def _checkpoint_epoch_number(path):
    match = re.search(r'epoch_(\d+)\.pt$', path)
    return int(match.group(1)) if match else -1


def save_checkpoint_with_rotation(model, optimizer, scheduler, epoch,
                                   epoch_loss, checkpoint_dir,
                                   max_to_keep, best_loss_so_far):
    os.makedirs(checkpoint_dir, exist_ok=True)

    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch_loss': epoch_loss,
        'num_train_pids': dataset.num_train_pids,
        'embedding_dim': cfg.embedding_dim,
    }

    # --- обычный чекпоинт текущей эпохи ---
    epoch_path = os.path.join(checkpoint_dir, f'epoch_{epoch:03d}.pt')
    torch.save(state, epoch_path)
    print(f'  Чекпоинт сохранён: {epoch_path}')

    # --- ротация: оставляем только max_to_keep последних по эпохе ---
    existing = sorted(
        glob.glob(os.path.join(checkpoint_dir, 'epoch_*.pt')),
        key=_checkpoint_epoch_number,
    )
    while len(existing) > max_to_keep:
        oldest = existing.pop(0)
        os.remove(oldest)
        print(f'  Удалён старый чекпоинт (ротация): {oldest}')

    # --- best checkpoint (никогда не ротируется) ---
    new_best_loss = best_loss_so_far
    if epoch_loss < best_loss_so_far:
        new_best_loss = epoch_loss
        best_path = os.path.join(checkpoint_dir, 'best_checkpoint.pt')
        torch.save(state, best_path)
        print(f'  Новый лучший чекпоинт (loss={epoch_loss:.4f}): {best_path}')

    return new_best_loss


def try_resume_from_checkpoint(model, optimizer, scheduler, checkpoint_dir):
    '''
    Ищет последний по номеру эпохи чекпоинт в checkpoint_dir. Если найден -
    загружает состояние модели/оптимизатора/scheduler'а.

    Возвращает:
        start_epoch: с какой эпохи продолжать (1, если чекпоинтов нет)
        best_loss_so_far: лучший loss из истории (inf, если чекпоинтов нет)
    '''
    if not os.path.isdir(checkpoint_dir):
        return 1, float('inf')

    existing = sorted(
        glob.glob(os.path.join(checkpoint_dir, 'epoch_*.pt')),
        key=_checkpoint_epoch_number,
    )
    if not existing:
        return 1, float('inf')

    latest_path = existing[-1]
    print(f'Найден чекпоинт для возобновления: {latest_path}')
    checkpoint = torch.load(latest_path, map_location=cfg.device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    print(f'Обучение будет продолжено с эпохи {start_epoch}.')

    # Лучший loss берём из best_checkpoint.pt, если он есть, иначе -
    # из loss последнего обычного чекпоинта (консервативная оценка).
    best_path = os.path.join(checkpoint_dir, 'best_checkpoint.pt')
    if os.path.isfile(best_path):
        best_state = torch.load(best_path, map_location='cpu')
        best_loss_so_far = best_state.get('epoch_loss', float('inf'))
    else:
        best_loss_so_far = checkpoint.get('epoch_loss', float('inf'))

    return start_epoch, best_loss_so_far


start_epoch, best_loss_so_far = try_resume_from_checkpoint(
    model, optimizer, scheduler, cfg.checkpoint_dir
)
print(f'Стартовая эпоха: {start_epoch}, лучший loss на данный момент: {best_loss_so_far}')

Найден чекпоинт для возобновления: /content/drive/MyDrive/VehicleReID/checkpoints/epoch_060.pt
Обучение будет продолжено с эпохи 61.
Стартовая эпоха: 61, лучший loss на данный момент: 1.3249129302638434


## Шаг 11. Цикл обучения

То же самое, что `train_one_epoch` в локальном `train.py`, но с
добавлением сохранения на Drive с ротацией после каждой эпохи и
поддержкой возобновления (`start_epoch` из предыдущего шага).

Если сессия Colab оборвётся, просто заново выполните все ячейки
ноутбука по порядку (Runtime -> Run all) — Шаг 10 автоматически найдёт
последний чекпоинт на Drive и обучение продолжится с нужной эпохи.

In [ ]:
import time


def train_one_epoch(model, dataloader, optimizer, ce_loss_fn, triplet_loss_fn,
                     triplet_weight, device, epoch, log_every):
    model.train()

    running_ce = 0.0
    running_triplet = 0.0
    running_correct = 0
    running_total = 0
    t_start = time.time()

    for it, (images, pids, camids, _paths) in enumerate(dataloader):
        images = images.to(device, non_blocking=True)
        pids = pids.to(device, non_blocking=True)

        embedding, logits, norm_embedding = model(images)

        ce_loss = ce_loss_fn(logits, pids)
        triplet_loss = triplet_loss_fn(norm_embedding, pids)
        loss = ce_loss + triplet_weight * triplet_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_ce += ce_loss.item()
        running_triplet += triplet_loss.item()
        preds = logits.argmax(dim=1)
        running_correct += (preds == pids).sum().item()
        running_total += pids.size(0)

        if (it + 1) % log_every == 0:
            avg_ce = running_ce / (it + 1)
            avg_triplet = running_triplet / (it + 1)
            acc = running_correct / running_total
            elapsed = time.time() - t_start
            print(
                f'[epoch {epoch}] iter {it + 1}/{len(dataloader)} | '
                f'CE: {avg_ce:.4f} | Triplet: {avg_triplet:.4f} | '
                f'ID-acc: {acc:.3f} | {elapsed:.1f}s'
            )

    avg_ce_final = running_ce / len(dataloader)
    avg_triplet_final = running_triplet / len(dataloader)
    return avg_ce_final + triplet_weight * avg_triplet_final


for epoch in range(start_epoch, cfg.epochs + 1):
    epoch_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        ce_loss_fn=ce_loss_fn,
        triplet_loss_fn=triplet_loss_fn,
        triplet_weight=cfg.triplet_loss_weight,
        device=cfg.device,
        epoch=epoch,
        log_every=cfg.log_every,
    )
    scheduler.step()

    current_lr = scheduler.get_last_lr()[0]
    print(f'== Эпоха {epoch}/{cfg.epochs} завершена. loss={epoch_loss:.4f}, lr={current_lr:.2e} ==')

    if epoch % cfg.save_every == 0 or epoch == cfg.epochs:
        best_loss_so_far = save_checkpoint_with_rotation(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            epoch_loss=epoch_loss,
            checkpoint_dir=cfg.checkpoint_dir,
            max_to_keep=cfg.max_checkpoints_to_keep,
            best_loss_so_far=best_loss_so_far,
        )

print('Обучение завершено.')

[epoch 58] iter 50/1155 | CE: 1.3729 | Triplet: 0.0224 | ID-acc: 0.964 | 19.2s
[epoch 58] iter 100/1155 | CE: 1.3847 | Triplet: 0.0220 | ID-acc: 0.966 | 38.4s
[epoch 58] iter 150/1155 | CE: 1.3813 | Triplet: 0.0210 | ID-acc: 0.967 | 57.4s
[epoch 58] iter 200/1155 | CE: 1.3728 | Triplet: 0.0204 | ID-acc: 0.968 | 75.9s
[epoch 58] iter 250/1155 | CE: 1.3714 | Triplet: 0.0201 | ID-acc: 0.969 | 94.5s
[epoch 58] iter 300/1155 | CE: 1.3692 | Triplet: 0.0205 | ID-acc: 0.968 | 113.2s
[epoch 58] iter 350/1155 | CE: 1.3660 | Triplet: 0.0206 | ID-acc: 0.970 | 132.1s
[epoch 58] iter 400/1155 | CE: 1.3611 | Triplet: 0.0203 | ID-acc: 0.971 | 150.8s
[epoch 58] iter 450/1155 | CE: 1.3600 | Triplet: 0.0205 | ID-acc: 0.970 | 169.5s
[epoch 58] iter 500/1155 | CE: 1.3579 | Triplet: 0.0206 | ID-acc: 0.971 | 188.3s
[epoch 58] iter 550/1155 | CE: 1.3549 | Triplet: 0.0202 | ID-acc: 0.972 | 207.0s
[epoch 58] iter 600/1155 | CE: 1.3531 | Triplet: 0.0199 | ID-acc: 0.973 | 225.7s
[epoch 58] iter 650/1155 | CE: 1.3

Оценка модели на валидационной выборке

In [17]:
!cp "/content/drive/MyDrive/VehicleReID/evaluate.py" /content/ml/evaluate.py
!python /content/ml/evaluate.py \
    --data-root /content/data/VeRi \
    --checkpoint /content/drive/MyDrive/VehicleReID/checkpoints/best_checkpoint.pt \
    --device cuda

Загружаю чекпоинт: /content/drive/MyDrive/VehicleReID/checkpoints/best_checkpoint.pt
  Эпоха чекпоинта: 59
Загружаю query/gallery (в порядке name_query.txt / name_test.txt)...
  query: 1678, gallery: 11579
Читаю gt_index.txt / jk_index.txt...
  Проверка индексации: совпадение pid при 1-based = 100.0%, при 0-based = 98.1%
  Используется индексация: 1-based
Извлекаю эмбеддинги query...
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processe

## Заметки на будущее

- **Отключение Colab по неактивности вкладки.** Держите вкладку браузера
  открытой либо периодически взаимодействуйте со страницей — Colab
  разрывает "простаивающие" сессии быстрее, чем сессии с лимитом по
  времени.
- **Лимит по времени бесплатного тарифа** — обычно около 12 часов
  непрерывной работы. При 60 эпохах и характерном времени ~2-4 минуты
  на эпоху на T4 (это будет заметно быстрее, чем ваши CPU-логи,
  благодаря GPU) обучение скорее всего уложится в один сеанс, но
  механизм возобновления на всякий случай уже встроен.
- **`evaluate.py` для честной оценки mAP/CMC** ещё предстоит перенести
  отдельно — сейчас в этом ноутбуке отслеживается только training loss,
  что подходит для контроля процесса обучения, но не заменяет
  полноценную валидацию на query/gallery сплитах.


## Ablation-эксперимент: RandomErasing на последних эпохах

Идея (обсуждали отдельно): агрессивные аугментации помогают на раннем и
среднем этапе обучения, но ближе к сходимости иногда мешают финальной
"калибровке" модели. Для ReID большинство аугментаций (ColorJitter,
RandomCrop, Flip) имитируют РЕАЛЬНЫЕ условия съёмки и трогать их не стоит.
Единственный кандидат на ослабление — `RandomErasing`, поскольку он
единственный создаёт искусственный шум, не соответствующий тому, что
модель увидит на реальном query-фото при инференсе.

**Важно про независимость чекпоинтов:**

  - Базовый `best_checkpoint.pt` (в `cfg.checkpoint_dir`) только
    ЧИТАЕТСЯ как стартовая точка — он никогда не перезаписывается кодом
    ниже.
  - Чекпоинты ablation-дообучения пишутся в **отдельную** папку на Drive
    (`checkpoints_ablation_random_erasing`), со своей независимой
    ротацией и своим `best_checkpoint.pt`.
  - Если ablation-сессия оборвётся, повторный запуск ячеек ниже сам
    найдёт последний **ablation**-чекпоинт и продолжит с него — точно
    так же, как это устроено для базового обучения.

Выполняйте эти ячейки ПОСЛЕ того, как базовое обучение (Шаги 1-11 выше)
полностью завершено — код ниже переиспользует объекты `dataset`,
`pk_sampler`, `batch_size`, `ce_loss_fn`, `triplet_loss_fn`, `cfg` и
функции `train_one_epoch` / `save_checkpoint_with_rotation` /
`try_resume_from_checkpoint`, созданные в предыдущих ячейках ноутбука.

### Конфигурация ablation-эксперимента

In [ ]:
ablation_cfg = SimpleNamespace(
    # Стартовая точка - лучший чекпоинт БАЗОВОГО обучения. Только читаем,
    # никогда не перезаписываем.
    base_best_checkpoint=os.path.join(cfg.checkpoint_dir, 'best_checkpoint.pt'),

    # Чекпоинты ablation-эксперимента - ОТДЕЛЬНАЯ папка на Drive, полностью
    # независимая от checkpoints/ базового обучения.
    checkpoint_dir=f'{DRIVE_PROJECT_ROOT}/checkpoints_ablation_random_erasing',
    max_checkpoints_to_keep=3,
    save_every=1,

    epochs=8,                  # дообучаем 5-10 эпох; 8 - разумная середина
    lr=cfg.lr * 0.1,            # для fine-tuning берём LR заметно меньше исходного
    weight_decay=cfg.weight_decay,
    triplet_loss_weight=cfg.triplet_loss_weight,

    # Главное отличие эксперимента: ослабляем/отключаем RandomErasing.
    # Остальные аугментации (ColorJitter, RandomCrop, Flip) оставляем как
    # в базовом обучении - они отражают реальные условия съёмки, а не
    # искусственный шум, который имеет смысл убирать к концу обучения.
    erasing_p=0.0,              # 0.0 = полностью отключить; можно попробовать 0.1

    log_every=cfg.log_every,
    device=cfg.device,
)

os.makedirs(ablation_cfg.checkpoint_dir, exist_ok=True)
print(ablation_cfg)

namespace(base_best_checkpoint='/content/drive/MyDrive/VehicleReID/checkpoints/best_checkpoint.pt', checkpoint_dir='/content/drive/MyDrive/VehicleReID/checkpoints_ablation_random_erasing', max_checkpoints_to_keep=3, save_every=1, epochs=8, lr=3.5000000000000004e-05, weight_decay=0.0005, triplet_loss_weight=1.0, erasing_p=0.0, log_every=50, device='cuda')


### Трансформации и DataLoader для ablation

Всё, кроме `RandomErasing`, идентично базовому обучению.

In [ ]:
def build_ablation_transform(image_size, erasing_p):
    h, w = image_size
    aug_list = [
        transforms.Resize((h, w)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.Pad(10),
        transforms.RandomCrop((h, w)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
    ]
    if erasing_p > 0:
        aug_list.append(transforms.RandomErasing(p=erasing_p, scale=(0.02, 0.2), ratio=(0.3, 3.3)))
    return transforms.Compose(aug_list)


ablation_transform = build_ablation_transform(cfg.image_size, ablation_cfg.erasing_p)

ablation_train_dataset = VeRiImageDataset(dataset.train, transform=ablation_transform)

ablation_train_loader = DataLoader(
    ablation_train_dataset,
    sampler=pk_sampler,            # тот же PK-сэмплер, что и в базовом обучении
    batch_size=batch_size,
    num_workers=cfg.num_workers,
    drop_last=True,
    pin_memory=(cfg.device == 'cuda'),
)

print('Ablation DataLoader готов. RandomErasing p =', ablation_cfg.erasing_p)

Ablation DataLoader готов. RandomErasing p = 0.0


### Модель ablation-эксперимента: старт от базового `best_checkpoint.pt`

Веса берутся из базового обучения, но оптимизатор и scheduler создаются
ЗАНОВО, с уменьшенным `lr` — это классический fine-tuning с чистого
листа по состоянию оптимизатора, а не продолжение исходного расписания.

Если у ablation-эксперимента уже есть свои собственные чекпоинты (после
обрыва сессии и повторного запуска), приоритет отдаётся им — веса из
базового обучения в этом случае заново не загружаются.

In [ ]:
ablation_model = VehicleReIDNet(
    num_classes=dataset.num_train_pids,
    embedding_dim=cfg.embedding_dim,
    pretrained=False,   # веса всё равно будут перезаписаны загруженным чекпоинтом
    dropout_p=cfg.dropout,
).to(ablation_cfg.device)

ablation_optimizer = torch.optim.Adam(
    ablation_model.parameters(), lr=ablation_cfg.lr, weight_decay=ablation_cfg.weight_decay
)
# Короткое дообучение - без дополнительного шага снижения LR внутри
# самого ablation-эксперимента (milestone далеко за пределами epochs).
ablation_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    ablation_optimizer, milestones=[10**9], gamma=1.0
)

ablation_start_epoch, ablation_best_loss = try_resume_from_checkpoint(
    ablation_model, ablation_optimizer, ablation_scheduler, ablation_cfg.checkpoint_dir
)

if ablation_start_epoch == 1:
    print(f'Загружаю стартовые веса из базового обучения: {ablation_cfg.base_best_checkpoint}')
    base_checkpoint = torch.load(ablation_cfg.base_best_checkpoint, map_location=ablation_cfg.device)
    ablation_model.load_state_dict(base_checkpoint['model_state_dict'])
    print(f"  (базовый чекпоинт был сохранён на эпохе {base_checkpoint.get('epoch', '?')} базового обучения)")
else:
    print(f'Найден собственный ablation-чекпоинт, продолжаю с эпохи {ablation_start_epoch}.')

print(f'Стартовая эпоха ablation: {ablation_start_epoch} / {ablation_cfg.epochs}')

Загружаю стартовые веса из базового обучения: /content/drive/MyDrive/VehicleReID/checkpoints/best_checkpoint.pt
  (базовый чекпоинт был сохранён на эпохе 59 базового обучения)
Стартовая эпоха ablation: 1 / 8


### Цикл дообучения (ablation)

Переиспользует `train_one_epoch` и `save_checkpoint_with_rotation` из
базового обучения без изменений — единственная разница уже "зашита" в
`ablation_train_loader` (другие аугментации) и `ablation_model` /
`ablation_optimizer` (независимые от базовых объекты).

In [ ]:
for epoch in range(ablation_start_epoch, ablation_cfg.epochs + 1):
    epoch_loss = train_one_epoch(
        model=ablation_model,
        dataloader=ablation_train_loader,
        optimizer=ablation_optimizer,
        ce_loss_fn=ce_loss_fn,
        triplet_loss_fn=triplet_loss_fn,
        triplet_weight=ablation_cfg.triplet_loss_weight,
        device=ablation_cfg.device,
        epoch=epoch,
        log_every=ablation_cfg.log_every,
    )
    ablation_scheduler.step()

    print(f'== [ABLATION] Эпоха {epoch}/{ablation_cfg.epochs} завершена. loss={epoch_loss:.4f} ==')

    if epoch % ablation_cfg.save_every == 0 or epoch == ablation_cfg.epochs:
        ablation_best_loss = save_checkpoint_with_rotation(
            model=ablation_model,
            optimizer=ablation_optimizer,
            scheduler=ablation_scheduler,
            epoch=epoch,
            epoch_loss=epoch_loss,
            checkpoint_dir=ablation_cfg.checkpoint_dir,
            max_to_keep=ablation_cfg.max_checkpoints_to_keep,
            best_loss_so_far=ablation_best_loss,
        )

print('Ablation-дообучение завершено.')
print('Базовые чекпоинты (checkpoints/) не были затронуты.')

[epoch 1] iter 50/1155 | CE: 1.3167 | Triplet: 0.0110 | ID-acc: 0.986 | 18.0s
[epoch 1] iter 100/1155 | CE: 1.3243 | Triplet: 0.0137 | ID-acc: 0.983 | 35.7s
[epoch 1] iter 150/1155 | CE: 1.3228 | Triplet: 0.0140 | ID-acc: 0.984 | 53.9s
[epoch 1] iter 200/1155 | CE: 1.3210 | Triplet: 0.0137 | ID-acc: 0.985 | 72.2s
[epoch 1] iter 250/1155 | CE: 1.3219 | Triplet: 0.0138 | ID-acc: 0.983 | 90.3s
[epoch 1] iter 300/1155 | CE: 1.3223 | Triplet: 0.0139 | ID-acc: 0.982 | 108.1s
[epoch 1] iter 350/1155 | CE: 1.3191 | Triplet: 0.0132 | ID-acc: 0.982 | 126.0s
[epoch 1] iter 400/1155 | CE: 1.3179 | Triplet: 0.0133 | ID-acc: 0.982 | 144.2s
[epoch 1] iter 450/1155 | CE: 1.3170 | Triplet: 0.0131 | ID-acc: 0.982 | 162.4s
[epoch 1] iter 500/1155 | CE: 1.3155 | Triplet: 0.0133 | ID-acc: 0.983 | 180.4s
[epoch 1] iter 550/1155 | CE: 1.3138 | Triplet: 0.0131 | ID-acc: 0.983 | 198.4s
[epoch 1] iter 600/1155 | CE: 1.3125 | Triplet: 0.0132 | ID-acc: 0.983 | 216.5s
[epoch 1] iter 650/1155 | CE: 1.3112 | Triplet

### Сравнение базового и ablation чекпоинтов по mAP/CMC

Требует, чтобы `evaluate.py` уже лежал на Drive (например, в
`{DRIVE_PROJECT_ROOT}/evaluate.py`) — копируем его в ту же структуру
модулей `/content/ml/`, что и остальной код, и запускаем как обычный
скрипт (не импортируем в ячейку) дважды: на базовом и на ablation
чекпоинте.

In [ ]:
!cp "{DRIVE_PROJECT_ROOT}/evaluate.py" /content/ml/evaluate.py

ablation_best_checkpoint = os.path.join(ablation_cfg.checkpoint_dir, 'best_checkpoint.pt')

print('=' * 70)
print('БАЗОВЫЙ чекпоинт (без изменений в аугментациях)')
print('=' * 70)
!python /content/ml/evaluate.py \
    --data-root {cfg.data_root} \
    --checkpoint {ablation_cfg.base_best_checkpoint} \
    --image-size {cfg.image_size[0]} {cfg.image_size[1]} \
    --device {cfg.device}

print()
print('=' * 70)
print(f'ABLATION чекпоинт (RandomErasing p={ablation_cfg.erasing_p}, {ablation_cfg.epochs} доп. эпох)')
print('=' * 70)
!python /content/ml/evaluate.py \
    --data-root {cfg.data_root} \
    --checkpoint {ablation_best_checkpoint} \
    --image-size {cfg.image_size[0]} {cfg.image_size[1]} \
    --device {cfg.device}

БАЗОВЫЙ чекпоинт (без изменений в аугментациях)
Загружаю чекпоинт: /content/drive/MyDrive/VehicleReID/checkpoints/best_checkpoint.pt
  Эпоха чекпоинта: 59
Загружаю query/gallery (в порядке name_query.txt / name_test.txt)...
  query: 1678, gallery: 11579
Читаю gt_index.txt / jk_index.txt...
  Проверка индексации: совпадение pid при 1-based = 100.0%, при 0-based = 98.1%
  Используется индексация: 1-based
Извлекаю эмбеддинги query...
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarnin

### Заметка по интерпретации

Сравнивайте mAP и Rank-1 из двух блоков вывода выше. Если ablation дал
заметный прирост (скажем, +1-2 п.п. mAP и больше) — это самостоятельный
результат для раздела экспериментов диплома: "ослабление RandomErasing
на последних эпохах дообучения улучшает генерализацию модели на X%".
Если прироста нет или он в пределах шума — тоже валидный вывод:
подтверждает, что аугментационная схема "Bag of Tricks" (постоянная
сила аугментаций на всём обучении) уже была близка к оптимуму для этой
задачи, и в тексте работы можно честно написать, что ablation не показал
значимого эффекта — это тоже часть научной добросовестности.

In [19]:
!python /content/ml/evaluate.py --data-root data/VeRi --checkpoint /content/drive/MyDrive/VehicleReID/checkpoints/best_checkpoint.pt --flip-tta --rerank

Загружаю чекпоинт: /content/drive/MyDrive/VehicleReID/checkpoints/best_checkpoint.pt
  Эпоха чекпоинта: 59
Загружаю query/gallery (в порядке name_query.txt / name_test.txt)...
  query: 1678, gallery: 11579
Читаю gt_index.txt / jk_index.txt...
  Проверка индексации: совпадение pid при 1-based = 100.0%, при 0-based = 98.1%
  Используется индексация: 1-based
Извлекаю эмбеддинги query...
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processe